In [1]:
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass
import os
import time
import numpy as np
import pandas as pd
import MetaTrader5 as mt5

In [2]:
mt5.initialize()

login = os.environ["FTMO_DEMO_LOGIN"]
password = os.environ["FTMO_DEMO_PASSWORD"]
server = os.environ["FTMO_DEMO_SERVER"]

mt5.login(login=login, password=password, server=server)

False

In [3]:
mt5.symbol_info("EURUSD")._asdict()

{'custom': False,
 'chart_mode': 0,
 'select': True,
 'visible': True,
 'session_deals': 0,
 'session_buy_orders': 0,
 'session_sell_orders': 0,
 'volume': 0,
 'volumehigh': 0,
 'volumelow': 0,
 'time': 1735343699,
 'digits': 5,
 'spread': 4,
 'spread_float': True,
 'ticks_bookdepth': 0,
 'trade_calc_mode': 0,
 'trade_mode': 4,
 'start_time': 0,
 'expiration_time': 0,
 'trade_stops_level': 0,
 'trade_freeze_level': 0,
 'trade_exemode': 2,
 'swap_mode': 1,
 'swap_rollover3days': 3,
 'margin_hedged_use_leg': False,
 'expiration_mode': 15,
 'filling_mode': 3,
 'order_mode': 127,
 'order_gtc_mode': 0,
 'option_mode': 0,
 'option_right': 0,
 'bid': 1.0424,
 'bidhigh': 1.04441,
 'bidlow': 1.0405,
 'ask': 1.04244,
 'askhigh': 1.04444,
 'asklow': 1.04053,
 'last': 0.0,
 'lasthigh': 0.0,
 'lastlow': 0.0,
 'volume_real': 0.0,
 'volumehigh_real': 0.0,
 'volumelow_real': 0.0,
 'option_strike': 0.0,
 'point': 1e-05,
 'trade_tick_value': 1.0,
 'trade_tick_value_profit': 1.0,
 'trade_tick_value_loss'

In [ ]:
TRADE_REQUEST_ACTIONS = [
    mt5.TRADE_ACTION_DEAL,      # Place an order for an instant deal with the specified parameters (set a market order)
    mt5.TRADE_ACTION_PENDING,   # Place an order for performing a deal at specified conditions (pending order)
    mt5.TRADE_ACTION_SLTP,      # Change open position Stop Loss and Take Profit
    mt5.TRADE_ACTION_MODIFY,    # Change parameters of the previously placed trading order
    mt5.TRADE_ACTION_REMOVE,    # Remove previously placed pending order
    mt5.TRADE_ACTION_CLOSE_BY,  # Close a position by an opposite one
]

ORDER_TYPE_FILLING = [
    mt5.ORDER_FILLING_FOK,      # This execution policy means that an order can be executed only in the specified volume. 
                                # If the necessary amount of a financial instrument is currently unavailable in the market, 
                                # the order will not be executed. The desired volume can be made up of several available offers.
    mt5.ORDER_FILLING_IOC,      # An agreement to execute a deal at the maximum volume available in the market within the volume 
                                # specified in the order. If the request cannot be filled completely, an order with the available 
                                # volume will be executed, and the remaining volume will be canceled.
    mt5.ORDER_FILLING_RETURN,   # This policy is used only for market (ORDER_TYPE_BUY and ORDER_TYPE_SELL), limit and stop limit 
                                # orders (ORDER_TYPE_BUY_LIMIT, ORDER_TYPE_SELL_LIMIT, ORDER_TYPE_BUY_STOP_LIMIT and 
                                # ORDER_TYPE_SELL_STOP_LIMIT) and only for the symbols with Market or Exchange execution modes. 
                                # If filled partially, a market or limit order with the remaining volume is not canceled, and is processed further.
]

ORDER_TYPE_TIME = [
    mt5.ORDER_TIME_GTC,
    mt5.ORDER_TIME_DAY,
    mt5.ORDER_TIME_SPECIFIED,
    mt5.ORDER_TIME_SPECIFIED_DAY
]

struct MqlTradeRequest
  {
   ENUM_TRADE_REQUEST_ACTIONS    action;           // Trade operation type
   ulong                         magic;            // Expert Advisor ID (magic number)
   ulong                         order;            // Order ticket
   string                        symbol;           // Trade symbol
   double                        volume;           // Requested volume for a deal in lots
   double                        price;            // Price
   double                        stoplimit;        // StopLimit level of the order
   double                        sl;               // Stop Loss level of the order
   double                        tp;               // Take Profit level of the order
   ulong                         deviation;        // Maximal possible deviation from the requested price
   ENUM_ORDER_TYPE               type;             // Order type
   ENUM_ORDER_TYPE_FILLING       type_filling;     // Order execution type
   ENUM_ORDER_TYPE_TIME          type_time;        // Order expiration type
   datetime                      expiration;       // Order expiration time (for the orders of ORDER_TIME_SPECIFIED type)
   string                        comment;          // Order comment
   ulong                         position;         // Position ticket
   ulong                         position_by;      // The ticket of an opposite position
  };

Market Order

In [3]:
request = {
    "action": mt5.TRADE_ACTION_DEAL,
    "symbol": "EURUSD",
    "volume": 0.02,
    "type": mt5.ORDER_TYPE_BUY,
    "type_filling": mt5.ORDER_FILLING_FOK,
}

result = mt5.order_send(request)
result

OrderSendResult(retcode=10009, deal=0, order=196193159, volume=0.02, price=0.0, bid=0.0, ask=0.0, comment='Request executed', request_id=2493602307, retcode_external=0, request=TradeRequest(action=1, magic=0, order=0, symbol='EURUSD', volume=0.02, price=0.0, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=0, type_filling=0, type_time=0, expiration=0, comment='', position=0, position_by=0))

In [55]:
@dataclass
class MT5Position:
    def __init__(self, mt5_position):
        self.datetime = mt5_position.time_msc

MT5Position(mt5.positions_get()[0])

MT5Position(mt5_position=TradePosition(ticket=196190782, time=1735308886, time_msc=1735308886718, time_update=1735308886, time_update_msc=1735308886718, type=0, magic=0, identifier=196190782, reason=3, volume=0.01, price_open=1.04395, sl=0.0, tp=0.0, price_current=1.04319, swap=0.0, profit=-0.76, symbol='EURUSD', comment='', external_id=''))

In [58]:
mt5.positions_get()[0]

TradePosition(ticket=196190782, time=1735308886, time_msc=1735308886718, time_update=1735308886, time_update_msc=1735308886718, type=0, magic=0, identifier=196190782, reason=3, volume=0.01, price_open=1.04395, sl=0.0, tp=0.0, price_current=1.04321, swap=0.0, profit=-0.74, symbol='EURUSD', comment='', external_id='')

In [10]:
result

OrderSendResult(retcode=10009, deal=0, order=196190782, volume=0.01, price=0.0, bid=0.0, ask=0.0, comment='Request executed', request_id=2493602306, retcode_external=0, request=TradeRequest(action=1, magic=0, order=0, symbol='EURUSD', volume=0.01, price=0.0, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=0, type_filling=0, type_time=0, expiration=0, comment='', position=0, position_by=0))

In [15]:
position = mt5.positions_get()[0]
print(datetime.fromtimestamp(position.time, tz=timezone.utc).replace(tzinfo=None))
position._asdict()

2024-12-28 18:56:57


{'ticket': 196301422,
 'time': 1735412217,
 'time_msc': 1735412217858,
 'time_update': 1735412217,
 'time_update_msc': 1735412217858,
 'type': 0,
 'magic': 0,
 'identifier': 196301422,
 'reason': 3,
 'volume': 0.01,
 'price_open': 94894.36,
 'sl': 0.0,
 'tp': 0.0,
 'price_current': 94760.04,
 'swap': 0.0,
 'profit': -1.34,
 'symbol': 'BTCUSD',
 'comment': '',
 'external_id': ''}

In [103]:
from_date=datetime.utcfromtimestamp(1735308886)
to_date=datetime.now(tz=timezone(timedelta(hours=2))).replace(tzinfo=None)
print(from_date, to_date)
mt5.history_orders_get(datetime(2024,12,1), to_date)

2024-12-27 14:14:46 2024-12-27 15:41:24.972592


(TradeOrder(ticket=195720479, time_setup=1734955737, time_setup_msc=1734955737081, time_done=1734955737, time_done_msc=1734955737318, time_expiration=0, type=1, type_time=0, type_filling=1, state=4, magic=0, position_id=195720479, position_by_id=0, reason=0, volume_initial=1.0, volume_current=0.0, price_open=0.0, sl=0.0, tp=0.0, price_current=1.04058, price_stoplimit=0.0, symbol='EURUSD', comment='', external_id='195720479-O'),
 TradeOrder(ticket=195721196, time_setup=1734956018, time_setup_msc=1734956018516, time_done=1734956018, time_done_msc=1734956018767, time_expiration=0, type=0, type_time=0, type_filling=1, state=4, magic=0, position_id=195720479, position_by_id=0, reason=4, volume_initial=1.0, volume_current=0.0, price_open=1.04084, sl=0.0, tp=0.0, price_current=1.04084, price_stoplimit=0.0, symbol='EURUSD', comment='[sl 1.04084]', external_id='195721196-O'),
 TradeOrder(ticket=195725749, time_setup=1734957658, time_setup_msc=1734957658723, time_done=1734957658, time_done_msc=1

In [4]:

class Position:
    def __init__(self, ticket, time, time_msc, time_update, time_update_msc, type, magic, identifier, reason, volume, price_open, sl, tp, price_current, swap, profit, symbol, comment, external_id):
        self.ticket = ticket
        self.time = time
        self.time_msc = time_msc
        self.time_update = time_update
        self.time_update_msc = time_update_msc
        self.type = type
        self.magic = magic
        self.identifier = identifier
        self.reason = reason
        self.volume = volume
        self.price_open = price_open
        self.sl = sl
        self.tp = tp
        self.price_current = price_current
        self.swap = swap
        self.profit = profit
        self.symbol = symbol
        self.comment = comment
        self.external_id = external_id

    @staticmethod
    def from_mt5_position(mt5_position):
        return Position(
            ticket=mt5_position.ticket,
            time=mt5_position.time,
            time_msc=mt5_position.time_msc,
            time_update=mt5_position.time_update,
            time_update_msc=mt5_position.time_update_msc,
            type=mt5_position.type,
            magic=mt5_position.magic,
            identifier=mt5_position.identifier,
            reason=mt5_position.reason,
            volume=mt5_position.volume,
            price_open=mt5_position.price_open,
            sl=mt5_position.sl,
            tp=mt5_position.tp,
            price_current=mt5_position.price_current,
            swap=mt5_position.swap,
            profit=mt5_position.profit,
            symbol=mt5_position.symbol,
            comment=mt5_position.comment,
            external_id=mt5_position.external_id
        )


# Private function to send trade requests
def _order_send(action, magic=None, order=None, symbol=None, volume=None, price=None, stoplimit=None, sl=None, tp=None, deviation=None, type=None, type_filling=None, type_time=None, expiration=None, comment="", position=None, position_by=None):
    request = {key: value for key, value in locals().items() if value is not None}
    result = mt5.order_send(request)
    return result
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print(f"OrderSend failed, retcode={result.retcode}")
        return None
    print(f"Operation successful, deal ID={result.deal}")
    # Retrieve the updated position from MetaTrader 5
    positions = mt5.positions_get(ticket=result.order)
    if positions:
        return Position.from_mt5_position(positions[0])
    return None

# Function to place a market order
def market_order(symbol, order_size, sl=None, tp=None, magic=0, comment=""):
    order_type = mt5.ORDER_TYPE_BUY if order_size > 0 else mt5.ORDER_TYPE_SELL
    return _order_send(
        action=mt5.TRADE_ACTION_DEAL, 
        symbol=symbol, 
        volume=abs(order_size), 
        type=order_type, 
        sl=sl, 
        tp=tp, 
        type_filling=mt5.ORDER_FILLING_FOK,
        magic=magic,
        comment=comment
    )

# Function to change SL/TP of a position
def change_position_sltp(position, sl=None, tp=None, magic=0, comment=""):
    return _order_send(
        action=mt5.TRADE_ACTION_SLTP,
        position=position.ticket,
        sl=sl,
        tp=tp,
        magic=magic,
        comment=comment
    )

def close_position(position, volume=None, magic=123456, comment=""):
    reverse_type = mt5.ORDER_TYPE_SELL if position.type == mt5.ORDER_TYPE_BUY else mt5.ORDER_TYPE_BUY

    # Create an opposite market order for the specified volume
    new_order_result = _order_send(
        action=mt5.TRADE_ACTION_DEAL,
        symbol=position.symbol,
        volume=volume or position.volume,
        type=reverse_type,
        magic=magic,
        comment=comment or "Close position"
    )

    print(position.ticket, new_order_result.order)
    time.sleep(0.1)
    
    if not new_order_result:
        print("Failed to create the opposite order for closing the position.")
        return None

    # Close the position using the created order
    return _order_send(
        action=mt5.TRADE_ACTION_CLOSE_BY,
        position=position.ticket,
        position_by=new_order_result.order,
    )

def close_all_positions(symbol=None):
    positions = mt5.positions_get()
    for position in positions:
        close_position(position)

In [162]:
close_all_positions()

196305744 196305781
196305747 196305783
196305773 196305785
196305774 196305787


In [8]:
symbol = "BTCUSD"
bid = mt5.symbols_get("BTCUSD")[0].bid
ask = mt5.symbols_get("BTCUSD")[0].ask
order_result = market_order("BTCUSD", 0.01)
positions = mt5.positions_get(ticket=order_result.order)
position = positions[0]
slippage = position.price_open-ask
print(bid, ask)
print(position.price_open, slippage)
print(order_result)
print(position)


94847.74 94882.36
94882.36 0.0
OrderSendResult(retcode=10009, deal=0, order=196309400, volume=0.01, price=0.0, bid=0.0, ask=0.0, comment='Request executed', request_id=1533340810, retcode_external=0, request=TradeRequest(action=1, magic=0, order=0, symbol='BTCUSD', volume=0.01, price=0.0, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=0, type_filling=0, type_time=0, expiration=0, comment='', position=0, position_by=0))
TradePosition(ticket=196309400, time=1735444146, time_msc=1735444146774, time_update=1735444146, time_update_msc=1735444146774, type=0, magic=0, identifier=196309400, reason=3, volume=0.01, price_open=94882.36, sl=0.0, tp=0.0, price_current=94847.74, swap=0.0, profit=-0.35, symbol='BTCUSD', comment='', external_id='')


In [7]:
order_result

OrderSendResult(retcode=10018, deal=0, order=0, volume=0.0, price=0.0, bid=0.0, ask=0.0, comment='Market closed', request_id=1533340809, retcode_external=0, request=TradeRequest(action=1, magic=0, order=0, symbol='BTCUSD', volume=0.01, price=0.0, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=0, type_filling=0, type_time=0, expiration=0, comment='', position=0, position_by=0))

In [137]:
#trailing stop
while True:
    bid = mt5.symbols_get("BTCUSD")[0].bid
    position = mt5.positions_get(ticket=order_result.order)[0]
    change_position_sltp(position, sl=max(bid-10, position.sl))
    time.sleep(0.1)

IndexError: tuple index out of range

In [91]:
close_position(position)

196304970 196304971


OrderSendResult(retcode=10009, deal=187788266, order=196304972, volume=0.01, price=95084.36, bid=0.0, ask=0.0, comment='Request executed', request_id=2023524026, retcode_external=0, request=TradeRequest(action=10, magic=0, order=0, symbol='', volume=0.0, price=0.0, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=0, type_filling=0, type_time=0, expiration=0, comment='', position=196304970, position_by=196304971))

In [138]:
position = mt5.positions_get()[0]

IndexError: tuple index out of range